Install lxml if needed

In [5]:
!pip install BeautifulSoup

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ------- -------------------------------- 0.8/4.0 MB 11.2 MB/s eta 0:00:01
   ---------------------------- ----------- 2.9/4.0 MB 7.3 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 7.3 MB/s  0:00:00


In [3]:
import pandas as pd
import os
from bs4 import BeautifulSoup

# CONFIGURATION
data_folder = "traffic data"  # folder containing your .htm files
timestep_minutes = 1          # simulation timestep in minutes
output_file = "traffic_metrics_per_timestep.csv"

# Vehicle types from your data
vehicle_columns = [
    'Heavy Truck', 'Medium Truck', 'Small Truck', 'Large Bus', 'Medium Bus',
    'Micro Bus', 'Utility', 'Car', 'Auto Rickshaw', 'Motor Cycle', 'Bi-Cycle',
    'Cycle Rickshaw', 'Cart', 'Motorized', 'Non Motorized'
]

# HELPER FUNCTION
def process_traffic_file(file_path):
    """Read a traffic.htm file and compute metrics per vehicle type (per timestep split at both ends)."""
    with open(file_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    # Find the main table containing traffic data
    main_table = soup.find("table", style="width:2500px;")
    if not main_table:
        raise ValueError(f"No main table found in {file_path}")
    inner_table = main_table.find("table")  # nested table with headers & data

    # Extract rows
    rows = inner_table.find_all("tr")

    # Find the header row containing vehicle types
    header_row = None
    for r in rows:
        cells = r.find_all("td")
        texts = [c.get_text(strip=True) for c in cells]
        if "Heavy Truck" in texts:
            header_row = texts
            break

    if not header_row:
        raise ValueError(f"No header row found in {file_path}")

    # Vehicle column indices (from header_row)
    start_idx = 9  # from your snippet, vehicle data starts at index 9
    end_idx = start_idx + len(vehicle_columns)

    # Extract data rows (rows after header_row)
    data_rows = []
    start = False
    for r in rows:
        cells = r.find_all("td")
        texts = [c.get_text(strip=True) for c in cells]
        if texts == header_row:
            start = True
            continue
        if start:
            if len(texts) >= end_idx:  # make sure it's a data row
                data_rows.append(texts[start_idx:end_idx])

    # Create DataFrame
    df = pd.DataFrame(data_rows, columns=vehicle_columns)
    df = df.apply(pd.to_numeric, errors='coerce').fillna(0)

    # Total daily count per vehicle type
    daily_totals = df.sum() / len(df)

    # Compute per-timestep injection, split across start and finish
    minutes_per_day = 24 * 60
    per_timestep_each_end = daily_totals / minutes_per_day * timestep_minutes / 2

    return per_timestep_each_end

# MAIN LOOP
all_metrics = []

for file_name in os.listdir(data_folder):
    if file_name.endswith(".htm"):
        file_path = os.path.join(data_folder, file_name)
        per_timestep_each_end = process_traffic_file(file_path)

        # Use only the first part of filename before dot
        road_name = file_name.split(".")[0]

        # Prepare row for output
        row = {"Road": road_name}
        for v in vehicle_columns:
            row[f"{v} per timestep (each end)"] = per_timestep_each_end[v]
        all_metrics.append(row)

# Save results to CSV
metrics_df = pd.DataFrame(all_metrics)
metrics_df.to_csv(output_file, index=False)
print(f"Per-timestep traffic metrics saved to {output_file}")

Per-timestep traffic metrics saved to traffic_metrics_per_timestep.csv
